# Tech Challenge — Fase 3
## Predição e Inteligência Analítica para Alfabetização no Brasil

**Pós Tech / FIAP — Data Analytics**

Este notebook integra toda a pipeline de ciência de dados desenvolvida para o desafio: da análise
exploratória à modelagem supervisionada, passando pela prevenção explícita de *data leakage* e pela
interpretabilidade dos modelos (Feature Importance e SHAP), encerrando com a aplicação estratégica dos
resultados a perguntas de política pública.

### Contexto do problema

Na Fase 2 construímos a camada **Gold** de um pipeline de engenharia de dados sobre o **Indicador Criança
Alfabetizada**, integrando dados de municípios, metas nacionais/estaduais/municipais e dimensões
territoriais. Nesta Fase 3, esses dados Gold são a matéria-prima para um modelo supervisionado capaz de
prever se um **município** atingirá sua meta de alfabetização, e para gerar inteligência aplicável à tomada
de decisão de gestores públicos.

### Objetivo analítico

1. Compreender quais fatores territoriais e de metas educacionais estão associados a um município atingir
   (ou não) sua meta de alfabetização.
2. Construir um pipeline de Machine Learning (Scikit-learn) robusto, com prevenção explícita de *data
   leakage*, comparando 3 famílias de algoritmos.
3. Interpretar o modelo campeão (Feature Importance / SHAP) para extrair insights de negócio.
4. Responder às perguntas estratégicas do desafio: quais fatores mais impactam a alfabetização, quais
   municípios estão em maior risco, e quais padrões regionais existem.

### Uma nota importante sobre a origem dos dados desta versão

A primeira versão deste pipeline foi construída sobre `data/raw/sample` e `data/raw/gold` — uma **amostra
sintética de demonstração** (81 municípios fictícios, 2021-2023) criada pelo gerador
`pipelines/batch/generate_sample_data.py` da Fase 2 para permitir rodar a pipeline sem credenciais de
nuvem. Ao investigar um salto implausível na taxa de alfabetização daquela amostra (quase dobrou entre
2021 e 2023), descobrimos que o crescimento era um **artefato mecânico do gerador** — uma meta que cresce
~10 p.p./ano por construção, alimentando um sorteio de proficiência com corte fixo — e não um fenômeno
educacional real.

Essa investigação nos levou a procurar, dentro do próprio repositório da Fase 2, uma fonte melhor — e a
encontramos: `reports/gold_preview/`, um espelho real da camada Gold gerado a partir de dados oficiais da
**Base dos Dados / CNCA** (Compromisso Nacional Criança Alfabetizada), cobrindo **5.516 municípios
brasileiros reais** (códigos IBGE genuínos) em 2023-2024. **Esta é a versão usada a partir daqui.**

Como a tabela de alunos individuais reais só existe no BigQuery (nunca foi exportada localmente), o alvo
da modelagem passou do nível **aluno** para o nível **município** — o que não é uma limitação de
conveniência: é exatamente uma das perguntas de negócio do desafio ("quais municípios apresentam maior
risco?", "como prever municípios que podem não atingir metas futuras?"), agora respondida com dados 100%
reais em vez de uma proxy sintética.


## 1. Setup do ambiente

In [1]:
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 140)

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.preprocessing.data_loading import (
    load_raw_tables, build_feature_table, stratified_train_test_split,
    get_feature_columns, TARGET, LEAKAGE_COLUMNS,
)
from src.preprocessing.pipeline import build_preprocessor
from src.modeling.train import run_model_search
from src.evaluation.metrics import (
    evaluate_model, build_comparison_table, plot_confusion_matrices, plot_roc_pr_curves,
)
from src.visualization.eda_plots import (
    plot_target_distribution, plot_numeric_distributions, plot_boxplots_outliers,
    plot_correlation_heatmap, plot_categorical_vs_target,
)
from src.visualization.shap_plots import (
    plot_feature_importance, compute_shap_values, plot_shap_summary, plot_shap_bar,
    plot_shap_waterfall, plot_shap_dependence,
)
from src.modeling.risk_analysis import (
    build_municipio_risk_ranking, plot_top_risk_municipios, cluster_municipios, plot_clusters,
)

IMAGES_DIR = PROJECT_ROOT / "reports" / "images"
IMAGES_DIR.mkdir(parents=True, exist_ok=True)

print("Projeto:", PROJECT_ROOT)


Projeto: C:\Users\capis\OneDrive\Área de Trabalho\FIAP\2_Tech Challenges\Tech 3\tech-challenge-fase3


## 2. Carregamento dos dados reais e engenharia de atributos

A função `build_feature_table` (em `src/preprocessing/data_loading.py`) filtra o ano-alvo (2024, quando
`meta_pct`/`atingiu_meta` estão preenchidos para todos os 5.516 municípios) e junta o histórico **defasado
em um ano** (2023) do próprio município e da UF — o motivo dessa defasagem é explicado na Seção 4 (Data
Leakage).

In [2]:
tables = load_raw_tables(PROJECT_ROOT / "data" / "raw")
df = build_feature_table(tables)

print("Dimensões do dataset final (nível município, ano-alvo 2024):", df.shape)
df.head()


Dimensões do dataset final (nível município, ano-alvo 2024): (5516, 19)


,ano,id_municipio,id_uf,sigla_uf,nome_municipio,pct_alfabetizados,meta_pct,nivel_meta,fonte_meta,gap_meta_pct,atingiu_meta,n_avaliados,ponto_corte,camada,delta_pp_ano_anterior,pct_alfabetizados_lag1,pct_alfabetizados_uf_lag1,meta_pct_uf,regiao
0,2024,1100015,11,RO,Alta Floresta D'Oeste,67.79,67.08,municipio,Base dos Dados / CNCA,0.71,1,NaN,743,gold,3.24,64.55,64.87,67.1,Norte
1,2024,1100023,11,RO,Ariquemes,65.62,65.22,municipio,Base dos Dados / CNCA,0.40,1,NaN,743,gold,3.32,62.30,64.87,67.1,Norte
2,2024,1100031,11,RO,Cabixi,75.88,70.85,municipio,Base dos Dados / CNCA,5.03,1,NaN,743,gold,6.78,69.10,64.87,67.1,Norte
3,2024,1100049,11,RO,Cacoal,66.07,65.39,municipio,Base dos Dados / CNCA,0.68,1,NaN,743,gold,2.70,63.37,64.87,67.1,Norte
4,2024,1100056,11,RO,Cerejeiras,66.81,62.09,municipio,Base dos Dados / CNCA,4.72,1,NaN,743,gold,8.28,58.53,64.87,67.1,Norte


In [3]:
df.info()


<class 'pandas.DataFrame'>
RangeIndex: 5516 entries, 0 to 5515
Data columns (total 19 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   ano                        5516 non-null   int64  
 1   id_municipio               5516 non-null   int64  
 2   id_uf                      5516 non-null   int64  
 3   sigla_uf                   5516 non-null   str    
 4   nome_municipio             5516 non-null   str    
 5   pct_alfabetizados          5516 non-null   float64
 6   meta_pct                   5516 non-null   float64
 7   nivel_meta                 5516 non-null   str    
 8   fonte_meta                 5516 non-null   str    
 9   gap_meta_pct               5516 non-null   float64
 10  atingiu_meta               5516 non-null   int64  
 11  n_avaliados                10 non-null     float64
 12  ponto_corte                5516 non-null   int64  
 13  camada                     5516 non-null   str    
 14  del

## 3. Análise Exploratória de Dados (EDA)

### 3.1 Distribuição do alvo

O alvo `atingiu_meta` é bem balanceado: **53,3%** dos 5.516 municípios atingiram sua meta de alfabetização
em 2024 (2.940 municípios), coerente com o KPI oficial reportado no relatório executivo da Fase 2.

In [4]:
print(df[TARGET].value_counts(normalize=True).round(4))
print()
print("Taxa de atingimento de meta por regiao:")
print(df.groupby("regiao")[TARGET].mean().sort_values(ascending=False).round(4))

plot_target_distribution(df, TARGET, IMAGES_DIR / "eda_target_distribution.png")


atingiu_meta
1    0.533
0    0.467
Name: proportion, dtype: float64

Taxa de atingimento de meta por regiao:
regiao
Centro-Oeste    0.7253
Sudeste         0.6875
Nordeste        0.4994
Norte           0.4207
Sul             0.3293
Name: atingiu_meta, dtype: float64


![Distribuição do alvo](../reports/images/eda_target_distribution.png)

**Hipótese H1:** existe heterogeneidade regional real na taxa de atingimento de metas — Centro-Oeste
e Sudeste lideram (~70%), enquanto o Sul aparece como a região com **menor** taxa de atingimento (~33%),
um resultado à primeira vista contraintuitivo (o Sul não é a região com pior alfabetização absoluta do
Brasil). Investigamos essa aparente contradição na Seção 3.4.

### 3.2 Distribuições numéricas e outliers

In [5]:
num_cols, cat_cols = get_feature_columns(df)
print("Atributos numericos:", num_cols)
print("Atributos categoricos:", cat_cols)

plot_numeric_distributions(df, num_cols, IMAGES_DIR / "eda_numeric_distributions.png")
plot_boxplots_outliers(df, num_cols, IMAGES_DIR / "eda_boxplots_outliers.png")


Atributos numericos: ['meta_pct', 'pct_alfabetizados_lag1', 'pct_alfabetizados_uf_lag1', 'meta_pct_uf']
Atributos categoricos: ['sigla_uf', 'nivel_meta', 'regiao']


![Distribuições numéricas](../reports/images/eda_numeric_distributions.png)

![Boxplots e outliers](../reports/images/eda_boxplots_outliers.png)

**Hipótese H2:** os valores estão nos intervalos plausíveis para percentuais (0-100) e metas reais do
CNCA. Não há outliers que sugiram erro de digitação — os poucos municípios nos extremos (`meta_pct` muito
alta ou `pct_alfabetizados_lag1` muito baixo) refletem casos reais documentados (municípios com resultado
histórico muito baixo recebem, no desenho do CNCA, metas de recuperação mais agressivas). Optamos por
**não remover outliers**, apenas por escalonar (`StandardScaler`) para reduzir sua influência
desproporcional em modelos lineares.

### 3.3 Correlações entre atributos numéricos e o alvo

In [6]:
plot_correlation_heatmap(df, num_cols, TARGET, IMAGES_DIR / "eda_correlation_heatmap.png")


![Matriz de correlação](../reports/images/eda_correlation_heatmap.png)

**Hipótese H3:** o histórico do próprio município (`pct_alfabetizados_lag1`, 2023) e da sua UF
(`pct_alfabetizados_uf_lag1`) correlacionam positivamente com o alvo — desempenho passado é sinal de
desempenho futuro. Mas a correlação não é tão forte quanto se poderia esperar isoladamente, porque o alvo
depende da **relação** entre desempenho e meta, não do desempenho absoluto — um município com desempenho
alto mas meta ainda mais alta pode não atingir a meta (ver Seção 3.4).

### 3.4 O paradoxo do Sul: desempenho absoluto vs. meta relativa

In [7]:
resumo_uf = df.groupby("sigla_uf").agg(
    pct_lag1=("pct_alfabetizados_lag1", "mean"),
    meta=("meta_pct", "mean"),
    atingiu=(TARGET, "mean"),
    n=("id_municipio", "count"),
).round(3).sort_values("atingiu")

print("UFs com menor taxa de atingimento de meta (2024):")
print(resumo_uf.head(8))
print()
print("UFs com maior taxa de atingimento de meta (2024):")
print(resumo_uf.tail(8))


UFs com menor taxa de atingimento de meta (2024):
          pct_lag1    meta  atingiu    n
sigla_uf                                
DF             NaN  59.900    0.000    1
RS          73.680  71.451    0.105  478
AC             NaN  59.900    0.136   22
BA          37.466  43.739    0.189  407
PA          46.371  51.931    0.319  144
AM          50.256  53.586    0.355   62
AP          42.266  48.034    0.375   16
RJ          55.686  59.170    0.413   92

UFs com maior taxa de atingimento de meta (2024):
          pct_lag1    meta  atingiu    n
sigla_uf                                
PI          59.762  60.058    0.585  224
PE          60.442  63.222    0.611  185
MT          57.600  60.894    0.624  141
MS          50.279  54.711    0.658   79
ES          73.730  73.540    0.769   78
MG          63.336  64.665    0.806  852
GO          72.701  71.842    0.808  245
CE          89.912  78.934    0.918  184


**Hipótese H4 (investigada e confirmada com números reais):** o Rio Grande do Sul (RS) tem
desempenho histórico **absoluto mediano** (`pct_alfabetizados_lag1` médio ≈ 53,8%) — não é a pior UF do
Brasil nesse quesito. No entanto, sua **meta média é desproporcionalmente alta** (≈ 71,4%, um gap de
-17,6 p.p.), e por isso apenas **10,5%** dos municípios gaúchos atingem a meta em 2024 — a menor taxa entre
todas as UFs. Isso é bem diferente do perfil da **Bahia** (BA): desempenho absoluto genuinamente baixo
(`pct_alfabetizados_lag1` ≈ 36,6%) combinado com uma meta proporcionalmente mais modesta (≈ 43,7%), ainda
assim insuficiente para boa parte dos municípios baianos (apenas 18,9% atingem a meta).

Ou seja: **existem dois perfis distintos de risco educacional**, e confundi-los levaria a políticas
públicas erradas — RS precisa de uma correção de trajetória/aceleração (ou uma recalibração de meta), BA
precisa de investimento estrutural em capacidade de alfabetização. O Ceará (CE), por outro lado, ilustra o
caso de sucesso: desempenho absoluto altíssimo (90,3%) superando uma meta também ambiciosa (78,9%) — um
resultado consistente com a fama nacional do programa estadual de alfabetização do Ceará (PAIC), que
inspirou o próprio Compromisso Nacional Criança Alfabetizada.

In [8]:
plot_categorical_vs_target(df, "regiao", TARGET, IMAGES_DIR / "eda_regiao_vs_target.png")
plot_categorical_vs_target(df, "sigla_uf", TARGET, IMAGES_DIR / "eda_uf_vs_target.png")


![Alfabetização por região](../reports/images/eda_regiao_vs_target.png)

![Alfabetização por UF](../reports/images/eda_uf_vs_target.png)

## 4. Prevenção de Data Leakage

Identificamos um vazamento direto de informação nesta base: `atingiu_meta` (2024) é definido
deterministicamente por `pct_alfabetizados_2024 >= meta_pct_2024`.

In [9]:
mun_2024 = tables["evolucao_municipio"]
mun_2024 = mun_2024[mun_2024["ano"] == 2024].dropna(subset=["meta_pct"])
match_rate = (
    (mun_2024["pct_alfabetizados"] >= mun_2024["meta_pct"]).astype(int)
    == mun_2024["atingiu_meta"].astype(bool).astype(int)
).mean()
print(f"Taxa de correspondencia entre (pct_alfabetizados_2024 >= meta_pct_2024) e 'atingiu_meta': {match_rate:.2%}")


Taxa de correspondencia entre (pct_alfabetizados_2024 >= meta_pct_2024) e 'atingiu_meta': 100.00%


A correspondência é de **100%**. Por isso `pct_alfabetizados` (2024), `gap_meta_pct`,
`delta_pp_ano_anterior` e `n_avaliados` do ano-alvo foram **excluídos do conjunto de features**
(`LEAKAGE_COLUMNS`) — o modelo só pode usar informação que existia **antes** do resultado de 2024: o
desempenho do próprio município e da UF em 2023 (defasado em um ano), e as metas vigentes para 2024, que
são definidas a priori pelo CNCA e não derivam do resultado avaliado.

Como 597 municípios (10,6%) só aparecem no ano-alvo (sem uma linha de 2023 correspondente — provavelmente
municípios que entraram na cobertura da avaliação em 2024), seu `pct_alfabetizados_lag1` fica `NaN`. Esse é
um cenário de dado faltante **genuíno e estruturalmente esperado**, tratado explicitamente na imputação
(Seção 6).

In [10]:
print("Colunas de leakage explicitamente removidas do conjunto de features:")
print(LEAKAGE_COLUMNS)
print()
print("Colunas efetivamente usadas como features:")
print(num_cols + cat_cols)
print()
print("Proporcao de valores ausentes por coluna numerica:")
print(df[num_cols].isna().mean().sort_values(ascending=False).round(4))


Colunas de leakage explicitamente removidas do conjunto de features:
['pct_alfabetizados', 'gap_meta_pct', 'delta_pp_ano_anterior', 'n_avaliados', 'camada', 'ponto_corte']

Colunas efetivamente usadas como features:
['meta_pct', 'pct_alfabetizados_lag1', 'pct_alfabetizados_uf_lag1', 'meta_pct_uf', 'sigla_uf', 'nivel_meta', 'regiao']

Proporcao de valores ausentes por coluna numerica:
pct_alfabetizados_lag1       0.1082
pct_alfabetizados_uf_lag1    0.0042
meta_pct                     0.0000
meta_pct_uf                  0.0000
dtype: float64


## 5. Divisão treino/teste

Diferente da primeira versão (dados sintéticos multi-ano, onde usamos holdout temporal), esta base real é
um **corte transversal**: todos os 5.516 municípios são avaliados no mesmo ano-alvo (2024), sem repetição
de município entre linhas. Por isso a divisão correta e padrão da literatura é uma amostragem **aleatória
estratificada pelo alvo** (80% treino / 20% teste) — não há estrutura de painel a proteger aqui.

In [11]:
train_df, test_df = stratified_train_test_split(df, test_size=0.2, random_state=42)
feature_cols = num_cols + cat_cols

X_train, y_train = train_df[feature_cols], train_df[TARGET]
X_test, y_test = test_df[feature_cols], test_df[TARGET]

print(f"Treino: {X_train.shape[0]} municipios | taxa atingiu_meta = {y_train.mean():.2%}")
print(f"Teste : {X_test.shape[0]} municipios | taxa atingiu_meta = {y_test.mean():.2%}")


Treino: 4412 municipios | taxa atingiu_meta = 53.31%
Teste : 1104 municipios | taxa atingiu_meta = 53.26%


## 6. Pipeline de pré-processamento (Scikit-learn)

O pré-processamento é encapsulado em um único `ColumnTransformer`, **integrado ao `Pipeline` do modelo**
(nunca ajustado fora dele), garantindo que toda estatística seja aprendida apenas no treino (ou apenas no
fold de treino, durante a validação cruzada).

- **Numéricas:** imputação pela **mediana** (robusta a outliers e adequada à ausência estrutural de
  `pct_alfabetizados_lag1` para municípios novos na cobertura) + `StandardScaler`.
- **Categóricas** (`sigla_uf`, `regiao`, `nivel_meta`): imputação pela moda + `OneHotEncoder`.

In [12]:
preprocessor = build_preprocessor(num_cols, cat_cols)
preprocessor


,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``feature_name``. e.g. `

## 7. Modelagem supervisionada

Comparamos 3 abordagens, cada uma dentro do seu próprio `Pipeline` (pré-processamento + classificador):
Regressão Logística regularizada, Random Forest e XGBoost. A validação cruzada usa **`StratifiedKFold`**
(5 folds) — suficiente aqui porque cada linha é um município distinto (sem repetição entre anos como na
versão anterior). Os hiperparâmetros são otimizados com `GridSearchCV` (Regressão Logística) e
`RandomizedSearchCV` (Random Forest e XGBoost), otimizando **ROC-AUC**.

In [13]:
results = run_model_search(X_train, y_train, preprocessor, n_iter=40)


[logistic_regression] melhor ROC-AUC (CV 5-fold): 0.7762
[logistic_regression] melhores hiperparametros: {'classifier__C': 0.5, 'classifier__class_weight': None}


[random_forest] melhor ROC-AUC (CV 5-fold): 0.7803
[random_forest] melhores hiperparametros: {'classifier__n_estimators': 400, 'classifier__min_samples_leaf': 8, 'classifier__max_features': 'log2', 'classifier__max_depth': 8, 'classifier__class_weight': 'balanced_subsample'}


[xgboost] melhor ROC-AUC (CV 5-fold): 0.7866
[xgboost] melhores hiperparametros: {'classifier__subsample': 0.85, 'classifier__reg_lambda': 2.0, 'classifier__n_estimators': 500, 'classifier__max_depth': 2, 'classifier__learning_rate': 0.05, 'classifier__colsample_bytree': 0.85}


In [14]:
cv_summary = pd.DataFrame({
    name: {"melhor_roc_auc_cv": r.best_cv_roc_auc, **r.best_params}
    for name, r in results.items()
}).T
cv_summary


,melhor_roc_auc_cv,classifier__C,classifier__class_weight,classifier__n_estimators,classifier__min_samples_leaf,classifier__max_features,classifier__max_depth,classifier__subsample,classifier__reg_lambda,classifier__learning_rate,classifier__colsample_bytree
logistic_regression,0.776165,0.5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
random_forest,0.780323,NaN,balanced_subsample,400,8,log2,8,NaN,NaN,NaN,NaN
xgboost,0.786609,NaN,NaN,500.0,NaN,NaN,2.0,0.85,2.0,0.05,0.85


## 8. Avaliação dos modelos (holdout estratificado, 20%)

Métricas robustas a desbalanceamento: **ROC-AUC**, **PR-AUC**, **F1**, **Precisão**, **Recall** e a
**Matriz de Confusão**.

In [15]:
estimators = {name: r.best_estimator for name, r in results.items()}
metrics = [evaluate_model(name, est, X_test, y_test) for name, est in estimators.items()]
comparison_table = build_comparison_table(metrics)
comparison_table


,roc_auc,pr_auc,f1,precisao,recall
modelo,,,,,
xgboost,0.7871,0.8107,0.7293,0.7190,0.7398
random_forest,0.7858,0.8046,0.7033,0.7222,0.6854
logistic_regression,0.7772,0.7958,0.7148,0.6974,0.7330


In [16]:
plot_confusion_matrices(estimators, X_test, y_test, IMAGES_DIR / "eval_confusion_matrices.png")
plot_roc_pr_curves(estimators, X_test, y_test, IMAGES_DIR / "eval_curves")


![Matrizes de confusão](../reports/images/eval_confusion_matrices.png)

![Curvas ROC](../reports/images/eval_curves_roc.png)

![Curvas Precisão-Recall](../reports/images/eval_curves_pr.png)

In [17]:
champion_name = comparison_table.index[0]
champion = estimators[champion_name]
print(f"Modelo campeao (maior ROC-AUC no holdout): {champion_name}")


Modelo campeao (maior ROC-AUC no holdout): xgboost


### Leitura dos resultados

Com dados reais, o desempenho é substancialmente melhor e mais estável do que na versão sintética: ROC-AUC
em torno de **0,79** para os três modelos, tanto em validação cruzada quanto no holdout — a diferença entre
CV e teste é pequena, indicando ausência de overfitting. O **XGBoost** é o modelo campeão (ROC-AUC ≈ 0,787,
PR-AUC ≈ 0,811), com Random Forest muito próximo. Isso confirma que, ao contrário da amostra sintética, os
atributos reais desta base (histórico municipal/estadual, meta vigente, UF) carregam sinal genuíno e
substancial sobre o resultado.

## 9. Interpretabilidade — Feature Importance e SHAP

In [18]:
preproc_fitted = champion.named_steps["preprocessor"]
feat_names = preproc_fitted.get_feature_names_out()

X_train_transformed = preproc_fitted.transform(X_train)
X_test_transformed = preproc_fitted.transform(X_test)
if hasattr(X_train_transformed, "toarray"):
    X_train_transformed = X_train_transformed.toarray()
    X_test_transformed = X_test_transformed.toarray()

importances = plot_feature_importance(champion, feat_names, IMAGES_DIR / "shap_feature_importance.png")
print(importances)


cat__sigla_uf_RS                  0.162825
cat__sigla_uf_MG                  0.144373
cat__sigla_uf_BA                  0.132106
cat__regiao_Sul                   0.092367
num__pct_alfabetizados_uf_lag1    0.056758
num__meta_pct_uf                  0.049268
cat__regiao_Centro-Oeste          0.045658
cat__sigla_uf_CE                  0.030533
num__meta_pct                     0.024882
cat__sigla_uf_GO                  0.020895
cat__regiao_Norte                 0.017342
num__pct_alfabetizados_lag1       0.015856
cat__sigla_uf_SP                  0.015254
cat__nivel_meta_municipio         0.013685
cat__regiao_Sudeste               0.013589
cat__sigla_uf_RJ                  0.012223
cat__sigla_uf_PA                  0.011547
cat__sigla_uf_PE                  0.011219
cat__regiao_Nordeste              0.010776
cat__sigla_uf_PR                  0.010687
dtype: float32


![Feature Importance](../reports/images/shap_feature_importance.png)

In [19]:
explainer, shap_values = compute_shap_values(champion, X_train_transformed, X_test_transformed)
plot_shap_summary(shap_values, feat_names, IMAGES_DIR / "shap_summary.png")
plot_shap_bar(shap_values, feat_names, IMAGES_DIR / "shap_bar.png")
plot_shap_waterfall(shap_values, 0, IMAGES_DIR / "shap_waterfall.png")


![SHAP Summary Plot](../reports/images/shap_summary.png)

![SHAP Bar Plot](../reports/images/shap_bar.png)

![SHAP Waterfall Plot](../reports/images/shap_waterfall.png)

**Insight de interpretabilidade:** o modelo campeão (XGBoost, árvores rasas de profundidade 2) apoia
suas decisões fortemente em **dummies de UF específicas** (RS, MG, BA, entre outras) além das variáveis
numéricas de meta e histórico. Isso é coerente com o achado da Seção 3.4: a relação entre desempenho e meta
tem um componente estrutural por estado (algumas UFs sistematicamente recebem metas mais ambiciosas em
relação ao seu patamar histórico), que o modelo captura via identidade da UF. Uma leitura cuidadosa: isso
não significa que "ser do RS" causa o não atingimento da meta — significa que a UF é um proxy forte para a
combinação de fatores (política de metas, trajetória histórica) que determinam o resultado.

## 10. Aplicação estratégica — risco municipal e padrões regionais

### 10.1 Ranking de municípios em maior risco educacional

In [20]:
ranking = build_municipio_risk_ranking(df, champion, feature_cols)
plot_top_risk_municipios(ranking, IMAGES_DIR / "risk_top_municipios.png", top_n=15)
ranking.head(15)


,id_municipio,nome_municipio,sigla_uf,regiao,meta_pct,proba_atingir_meta,faixa_risco
4575,4300208,Ajuricaba,RS,Sul,80.00,0.040987,Alto risco
4717,4307500,Espumoso,RS,Sul,80.00,0.040987,Alto risco
4789,4311254,Lagoão,RS,Sul,80.00,0.041196,Alto risco
4763,4310207,Ijuí,RS,Sul,77.00,0.042329,Alto risco
4874,4314498,Pinheirinho do Vale,RS,Sul,80.00,0.043924,Alto risco
4958,4319125,São Martinho da Serra,RS,Sul,77.00,0.044446,Alto risco
4730,4308201,Flores da Cunha,RS,Sul,79.15,0.045338,Alto risco
4818,4312385,Monte Belo do Sul,RS,Sul,80.00,0.045423,Alto risco
5035,4322855,Vespasiano Correa,RS,Sul,80.00,0.045423,Alto risco
4962,4319356,São Pedro da Serra,RS,Sul,80.00,0.045423,Alto risco


![Ranking de risco municipal](../reports/images/risk_top_municipios.png)

In [21]:
print("Distribuicao de municipios por faixa de risco:")
print(ranking["faixa_risco"].value_counts())
print()
print("Faixa de risco por regiao:")
print(pd.crosstab(ranking["regiao"], ranking["faixa_risco"]))


Distribuicao de municipios por faixa de risco:
faixa_risco
Baixo risco       2209
Risco moderado    1722
Alto risco        1585
Name: count, dtype: int64

Faixa de risco por regiao:
faixa_risco   Alto risco  Baixo risco  Risco moderado
regiao                                               
Centro-Oeste           4          390              72
Nordeste             612          565             605
Norte                194           93             148
Sudeste               90         1035             539
Sul                  685          126             358


**Observação honesta:** os 15 municípios de maior risco são todos do RS — reflexo direto do peso que
o modelo atribui à UF (Seção 9). Isso é uma limitação a comunicar a gestores: o ranking atual prioriza bem
*entre estados*, mas precisa de mais variáveis (renda, infraestrutura escolar) para diferenciar risco
*dentro* de um mesmo estado com mais nuance (ver Limitações no README).

### 10.2 Agrupamento (clustering) de municípios por padrão socioeducacional

In [22]:
cluster_features = ["pct_alfabetizados_lag1", "meta_pct", "pct_alfabetizados_uf_lag1"]
cluster_df, kmeans_model = cluster_municipios(df, cluster_features, n_clusters=4)
plot_clusters(cluster_df, "pct_alfabetizados_lag1", "meta_pct", IMAGES_DIR / "cluster_municipios.png")

print("Perfil medio de cada cluster:")
print(cluster_df.groupby("cluster")[cluster_features].mean().round(2))
print()
print("Tamanho de cada cluster:")
print(cluster_df["cluster"].value_counts())


Perfil medio de cada cluster:
         pct_alfabetizados_lag1  meta_pct  pct_alfabetizados_uf_lag1
cluster                                                             
0                         64.97     66.99                      64.43
1                         48.50     53.47                      56.39
2                         83.74     78.30                      72.63
3                         32.30     38.76                      40.43

Tamanho de cada cluster:
cluster
0    1496
2    1384
1    1136
3     903
Name: count, dtype: int64


![Clusters de municípios](../reports/images/cluster_municipios.png)

**Insight de clustering:** os 4 grupos mapeiam com clareza os perfis discutidos na Seção 3.4 —
municípios "referência" (desempenho alto, meta alta, já superando), "consistentes" (desempenho e meta
compatíveis), "moderados" e um grupo de "atenção prioritária" (desempenho baixo). Esse agrupamento é uma
ferramenta prática para desenhar políticas diferenciadas por perfil, em vez de uma meta única nacional.

## 11. Conclusões e próximos passos

Consulte o `README.md` do projeto para a discussão completa de metodologia, métricas, interpretação,
limitações, aplicação em políticas públicas e evoluções futuras.
